# Клиенты ТЭ за июль 2026: три подсчёта

Зерно — договор. Три числа:

1. **final_df.** Актуальный файл: `Проверки в пространстве/final_script_2_1.ipynb`, функция `compute_final_df_for_month`, секция `01_sa_perimeter`. Старый `final_script_2.ipynb` не используется. Договор SA пересекает июль: открыт не позже 31.07 и закрыт не раньше 01.07 (или не закрыт). Привязка `P` тоже пересекает июль, но закрытие привязки строгое: `d_valid_to > 2026-07-01`. Дальше витрина только левыми джойнами добавляет атрибуты, число договоров не режет.
2. **Скрипт сборки ТЭ.** Файл `Проверки в пространстве/trade_acquiring_clients_month_excel.ipynb`, лист «С ТЭ». Договор и привязка `P` оба живы в конкретный день. Для июля этот день — 31.07.2026: `d_valid_to` пустая или `>= 2026-07-31`.
3. **Excel.** `/home/jovyan/documents/Equaring/Data/07_Июль_2026.xlsx`. Число — уникальные ID договора. Рядом печатается число строк файла.

Запустите ячейки по порядку.


In [ ]:
import getpass

import pandas as pd
from rail_connectors.connection import connect

MONTH_START = '2026-07-01'
MONTH_END = '2026-07-31'
EXCEL_JUL = '/home/jovyan/documents/Equaring/Data/07_Июль_2026.xlsx'

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
sql_final_df = f'''
SELECT count(DISTINCT a.n_agr) AS contracts
FROM ods_alpha.scd1_agreements a
JOIN ods_alpha.scd1_companies c
  ON c.n_cmp = a.n_cmp_client
WHERE upper(trim(cast(a.acq_class AS string))) = 'SA'
  AND cast(a.d_valid_from AS date) <= cast('{MONTH_END}' AS date)
  AND (a.d_valid_to IS NULL OR cast(a.d_valid_to AS date) >= cast('{MONTH_START}' AS date))
  AND coalesce(a.ods_deleted_flg, '0') <> '1'
  AND coalesce(c.ods_deleted_flg, '0') <> '1'
  AND c.c_inn IS NOT NULL
  AND EXISTS (
    SELECT 1
    FROM ods_alpha.scd1_agr_terms t
    WHERE cast(t.n_agr AS string) = cast(a.n_agr AS string)
      AND cast(t.d_valid_from AS date) <= cast('{MONTH_END}' AS date)
      AND (t.d_valid_to IS NULL OR cast(t.d_valid_to AS date) > cast('{MONTH_START}' AS date))
      AND upper(trim(cast(t.cf_ter_type AS string))) = 'P'
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
  )
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    n_final = int(imp.fetch(sql_final_df)['contracts'].iloc[0])
print('final_df, договоры за июль:', n_final)
n_final


In [ ]:
sql_script = f'''
SELECT count(DISTINCT a.n_agr) AS contracts
FROM ods_alpha.scd1_agreements a
JOIN ods_alpha.scd1_companies c
  ON c.n_cmp = a.n_cmp_client
WHERE upper(trim(a.acq_class)) = 'SA'
  AND a.d_valid_from <= '{MONTH_END}'
  AND (a.d_valid_to IS NULL OR a.d_valid_to >= '{MONTH_END}')
  AND coalesce(a.ods_deleted_flg, '0') <> '1'
  AND coalesce(c.ods_deleted_flg, '0') <> '1'
  AND c.c_inn IS NOT NULL
  AND EXISTS (
    SELECT 1
    FROM ods_alpha.scd1_agr_terms t
    WHERE t.n_agr = a.n_agr
      AND t.d_valid_from <= '{MONTH_END}'
      AND (t.d_valid_to IS NULL OR t.d_valid_to >= '{MONTH_END}')
      AND upper(trim(t.cf_ter_type)) = 'P'
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
  )
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    n_script = int(imp.fetch(sql_script)['contracts'].iloc[0])
print('скрипт сборки ТЭ, договоры на 31.07.2026:', n_script)
n_script


In [ ]:
def norm_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s.endswith('.0'):
        s = s[:-2]
    if s in {'', 'nan', 'None'}:
        return None
    return s

def pick(columns, names):
    low = {str(c).replace('\n', ' ').strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in low:
            return low[name.lower()]
    return None

raw = pd.read_excel(EXCEL_JUL, header=None, nrows=12)
header = 0
for i, row in raw.iterrows():
    cells = ' '.join(str(x).lower().replace('\n', ' ') for x in row.tolist() if pd.notna(x))
    if any(m in cells for m in ('аур', 'амортизац', 'фин. рез', 'фин.рез', 'фин рез')):
        header = i
        break
ex = pd.read_excel(EXCEL_JUL, header=header)
id_col = pick(ex.columns, ['ID договора', 'agr_id', 'abs_agr_id'])
num_col = pick(ex.columns, ['Номер договора'])
print('заголовок Excel, строка', header)
print('строк в файле:', len(ex))
if id_col:
    n_excel = int(ex[id_col].map(norm_id).nunique(dropna=True))
    print('уникальных ID договора (%s):' % id_col, n_excel)
elif num_col:
    n_excel = int(ex[num_col].map(norm_id).nunique(dropna=True))
    print('колонки ID договора нет, уникальных номеров (%s):' % num_col, n_excel)
else:
    n_excel = len(ex)
    print('колонки договора нет, беру число строк:', n_excel)
    print(list(ex.columns))
n_excel


In [ ]:
summary = pd.DataFrame([
    ['final_df, договоры за июль', n_final],
    ['скрипт сборки ТЭ, на 31.07.2026', n_script],
    ['Excel, уникальные договоры', n_excel],
], columns=['способ', 'договоров'])
summary
